# Hands-on Lab #4 – Support Vector Machines (SVM)

**Guided dataset:** [Wine Recognition – Scikit-learn](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_wine.html)  

**Objective:** Train and evaluate SVM classifiers, visualize decision boundaries and support vectors, compare linear, polynomial and RBF kernels, and tune `C` and `gamma`.

**Submission:** One `.ipynb` containing executed code, metrics, plots, the individual challenge, and conclusions.

## 1. Load and explore the Wine dataset
The dataset has 178 observations, 13 chemical features, and three wine classes.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.inspection import DecisionBoundaryDisplay
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, classification_report, ConfusionMatrixDisplay)

wine = load_wine()
X = pd.DataFrame(wine.data, columns=wine.feature_names)
y = wine.target
print('Shape:', X.shape)
print('Classes:', wine.target_names)
print('Distribution:', np.bincount(y))
display(X.head())

## 2. Split the data
Use a stratified 80/20 train/test split. Scaling will be applied inside pipelines to prevent leakage.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print('Training:', X_train.shape, '| Test:', X_test.shape)

## 3. Train a linear SVM

In [ ]:
svm_linear = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(kernel='linear', C=1))
])
svm_linear.fit(X_train, y_train)
y_pred = svm_linear.predict(X_test)
print(classification_report(y_test, y_pred, target_names=wine.target_names))
print('Support vectors per class:', svm_linear.named_steps['svm'].n_support_)

## 4. Evaluate performance

In [ ]:
print('Accuracy:', accuracy_score(y_test, y_pred))
print('Precision (macro):', precision_score(y_test, y_pred, average='macro'))
print('Recall (macro):', recall_score(y_test, y_pred, average='macro'))
print('F1 (macro):', f1_score(y_test, y_pred, average='macro'))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred,
    display_labels=wine.target_names, cmap='Blues')
plt.title('Linear SVM – Confusion Matrix')
plt.show()

## 5. Visualize decision boundaries and support vectors
**Note:** This separate visualization model uses only `alcohol` and `color_intensity` and is trained on the full dataset for illustration. It is **not** the 13-feature evaluation model. In multiclass SVM, multiple pairwise classifiers contribute to the boundaries.

In [ ]:
X_2d = X[['alcohol', 'color_intensity']]
scaler_2d = StandardScaler()
X_scaled_2d = scaler_2d.fit_transform(X_2d)
model_2d = SVC(kernel='linear', C=1)
model_2d.fit(X_scaled_2d, y)
fig, ax = plt.subplots(figsize=(8, 6))
DecisionBoundaryDisplay.from_estimator(
    model_2d, X_scaled_2d, response_method='predict',
    cmap='Pastel1', alpha=0.5, ax=ax
)
ax.scatter(X_scaled_2d[:, 0], X_scaled_2d[:, 1], c=y,
           cmap='viridis', edgecolors='black', s=35)
ax.scatter(model_2d.support_vectors_[:, 0],
           model_2d.support_vectors_[:, 1], s=150,
           facecolors='none', edgecolors='red', linewidths=1.8,
           label='Support vectors')
ax.set_xlabel('Alcohol (standardized)')
ax.set_ylabel('Color intensity (standardized)')
ax.set_title('SVM Decision Regions and Support Vectors')
ax.legend()
plt.show()

## 6. Compare kernels
Use macro-averaged metrics because the problem has three classes.

In [ ]:
results = []
for kernel in ['linear', 'poly', 'rbf']:
    model = Pipeline([
        ('scaler', StandardScaler()),
        ('svm', SVC(kernel=kernel, C=1))
    ])
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    results.append({
        'Kernel': kernel,
        'Accuracy': accuracy_score(y_test, pred),
        'Precision': precision_score(y_test, pred, average='macro', zero_division=0),
        'Recall': recall_score(y_test, pred, average='macro', zero_division=0),
        'F1-score': f1_score(y_test, pred, average='macro', zero_division=0),
        'Support vectors': len(model.named_steps['svm'].support_)
    })
results_df = pd.DataFrame(results)
display(results_df)

## 7. Tune `C` and `gamma` with cross-validation
Only the training set is used for hyperparameter selection. The test set is evaluated once after selection.

In [ ]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(kernel='rbf'))
])
param_grid = {
    'svm__C': [0.1, 1, 10, 100],
    'svm__gamma': [0.001, 0.01, 0.1, 1]
}
grid = GridSearchCV(pipeline, param_grid, cv=5,
                    scoring='f1_macro', n_jobs=-1)
grid.fit(X_train, y_train)
print('Best parameters:', grid.best_params_)
print('Best CV F1-macro:', grid.best_score_)
best_pred = grid.best_estimator_.predict(X_test) 
print('Test-set performance:')
print(classification_report(y_test, best_pred, target_names=wine.target_names))



## 8. Visualize the hyperparameter search

In [ ]:
cv_results = pd.DataFrame(grid.cv_results_)
score_table = cv_results.pivot(index='param_svm__C',
                               columns='param_svm__gamma',
                               values='mean_test_score')
fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(score_table.to_numpy(dtype=float), cmap='viridis', aspect='auto')
ax.set_xticks(range(len(score_table.columns)), labels=[str(x) for x in score_table.columns])
ax.set_yticks(range(len(score_table.index)), labels=[str(x) for x in score_table.index])
ax.set_xlabel('gamma')
ax.set_ylabel('C')
ax.set_title('Cross-validation F1-macro – RBF SVM')
for i in range(len(score_table.index)):
    for j in range(len(score_table.columns)):
        ax.text(j, i, f'{score_table.iloc[i, j]:.3f}',
                ha='center', va='center', color='white')
fig.colorbar(im, ax=ax, label='F1-macro')
plt.tight_layout()
plt.show()

## 9. Statement

**Individual Challenge:** Apply the same methodology to the [**Banknote Authentication dataset**](https://archive.ics.uci.edu/dataset/267/banknote+authentication) from the UCI Machine Learning Repository to classify banknotes using SVM.


In [ ]:
# TODO: Load Banknote Authentication from UCI
banknote_url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00267/data_banknote_authentication.txt'
columns = ['variance', 'skewness', 'curtosis', 'entropy', 'class']

# banknote_df = ...
# X_banknote = ...
# y_banknote = ...

# TODO: Train/test split and build a StandardScaler + SVC pipeline
# TODO: Compare linear and RBF kernels
# TODO: Compute metrics and plot a confusion matrix
# TODO: Tune C and gamma with GridSearchCV
